# Governed RAG Pipeline with TealTiger

This cookbook shows how to add deterministic governance guardrails to a Haystack RAG pipeline using [TealTiger](https://haystack.deepset.ai/integrations/tealtiger).

You'll learn how to:
- Scan queries for PII before they reach the LLM
- Enforce cost budgets per session
- Detect secrets in generated responses
- Produce structured audit receipts for compliance

All governance runs deterministically (regex + policy rules) with no LLM in the governance path and under 2ms overhead.

In [ ]:
!pip install haystack-ai haystack-tealtiger

## Set up the Governed RAG Pipeline

We'll build a simple RAG pipeline with an in-memory document store and wrap it with TealTiger governance checks at the query boundary.

In [ ]:
import os
from haystack import Pipeline, Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OpenAIGenerator

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# Create a document store with sample documents
document_store = InMemoryDocumentStore()
documents = [
    Document(content="TealTiger is an open-source AI governance SDK that provides deterministic policy enforcement for AI agents."),
    Document(content="Haystack is a production-ready framework for building RAG pipelines and AI applications."),
    Document(content="PII detection scans text for sensitive data like SSNs (e.g., 123-45-6789), credit cards, and email addresses."),
    Document(content="The company's annual revenue was $4.2M in 2025, with a net profit margin of 12%."),
]
document_store.write_documents(documents)

## Add TealTiger Governance

We'll configure governance policies:
- **PII block**: Deny queries containing SSNs, credit cards, or email addresses
- **Cost limit**: Cap session cost at $0.50
- **Secret detection**: Block responses containing API keys or tokens

In [ ]:
from haystack_tealtiger import TealTigerGovernanceChecker
from tealtiger import GovernancePolicy, GovernanceMode

# Configure governance policies
governance = TealTigerGovernanceChecker(
    policies=[
        GovernancePolicy.pii_block(["ssn", "credit_card", "email"]),
        GovernancePolicy.cost_limit(max_per_session=0.50),
        GovernancePolicy.secret_detection(),
    ],
    mode=GovernanceMode.ENFORCE,  # Block violating requests
)

## Build the Pipeline

The governance checker runs as a component in the pipeline. It scans the query before it reaches the retriever/LLM.

In [ ]:
# Build the RAG pipeline with governance
template = """
Given the following context, answer the question.

Context:
{% for doc in documents %}
- {{ doc.content }}
{% endfor %}

Question: {{ query }}
Answer:
"""

pipe = Pipeline()
pipe.add_component("governance", governance)
pipe.add_component("retriever", InMemoryBM25Retriever(document_store=document_store))
pipe.add_component("prompt_builder", PromptBuilder(template=template))
pipe.add_component("llm", OpenAIGenerator(model="gpt-4o-mini"))

# Connect components
pipe.connect("governance.query", "retriever.query")
pipe.connect("retriever", "prompt_builder.documents")
pipe.connect("governance.query", "prompt_builder.query")
pipe.connect("prompt_builder", "llm")

## Run a Clean Query (Allowed)

This query has no PII and is within budget — governance allows it through.

In [ ]:
# This query is clean — no PII, within budget
result = pipe.run({"governance": {"query": "What is TealTiger?"}})
print("Answer:", result["llm"]["replies"][0])
print("\nGovernance decision:", governance.last_decision)

## Run a Query with PII (Blocked)

This query contains an SSN — governance will block it in ENFORCE mode.

In [ ]:
# This query contains an SSN — governance will block it
try:
    result = pipe.run({"governance": {"query": "Look up records for SSN 123-45-6789"}})
except Exception as e:
    print(f"Blocked: {e}")
    print(f"Reason: {governance.last_decision.reason_codes}")
    print(f"Risk score: {governance.last_decision.risk_score}")

## Inspect the Audit Trail

Every governance decision produces a structured TEEC receipt — useful for SOC2/HIPAA compliance evidence.

In [ ]:
# View all governance decisions from this session
for i, decision in enumerate(governance.decisions):
    print(f"  Decision {i+1}: [{decision.action}] "
          f"reason={decision.reason_codes} "
          f"risk={decision.risk_score} "
          f"latency={decision.evaluation_time_ms:.2f}ms")

## Switch to MONITOR Mode (Dry Run)

In MONITOR mode, governance evaluates policies and records decisions but never blocks — useful for rolling out governance without risk.

In [ ]:
# Switch to MONITOR mode — logs violations but allows everything through
governance.mode = GovernanceMode.MONITOR

result = pipe.run({"governance": {"query": "Look up SSN 123-45-6789"}})
print("Query allowed through (MONITOR mode)")
print(f"Decision recorded: {governance.last_decision.action}")
print(f"Would have blocked: {governance.last_decision.reason_codes}")

## Summary

| Mode | Behavior |
|------|----------|
| **ENFORCE** | Evaluates policies, blocks violations |
| **MONITOR** | Evaluates policies, records decisions, allows all through (dry run) |
| **OBSERVE** | Skips evaluation, passes through with minimal audit |

**Resources:**
- [TealTiger Integration Page](https://haystack.deepset.ai/integrations/tealtiger)
- [TealTiger Docs](https://docs.tealtiger.ai)
- [GitHub](https://github.com/agentguard-ai/tealtiger)
- [PyPI](https://pypi.org/project/haystack-tealtiger/)